# Análise dos resultados do ataque ALGEN sobre embeddings DPED (SST-2)

Este notebook agrega e visualiza os resultados do ataque de **alinhamento X→Y** (ridge/pseudoinversa) com decodificador ALGEN **congelado**, em função de:

- **ε** (orçamento de privacidade do DPED) — e o baseline `bert_pretrained` (embeddings sem privacidade);
- **n** (`train_samples`): número de pares (frase, vetor) vazados usados para ajustar o alinhamento.

**Fonte dos dados:** `results_attack/<condição>/n<N>_r<R>/{summary.json, sentence_results.csv}`. A leitura é recursiva e não depende dos nomes das pastas: `epsilon`, `source` e `train_samples` vêm do conteúdo dos arquivos (o nome da pasta é só fallback).

**Escalas:** BLEU (`bleu`, `bleu1..4`) está em 0–100; ROUGE, `exact_match` e cosseno estão em 0–1.

**Roteiro**
1. Carga e validação dos dados
2. Engenharia de atributos (comprimento, sobreposição lexical, linha de base de acaso)
3. Panorama: métricas × (ε, n)
4. Tendências por n e por ε (com IC bootstrap)
5. Alinhamento (cosseno) vs qualidade textual
6. Distribuições e testes pareados
7. Tamanho da frase
8. Nível de palavra: o que é recuperado
9. Diversidade e colapso de modo
10. Dificuldade por frase e exemplos
11. Trade-off privacidade × vazamento (regressão, decomposição de variância, n mínimo)
12. DPED vs baseline BERT
13. Achados automáticos e exportação

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
import os, re, json, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, NullLocator
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

RESULTS_DIR = os.environ.get('RESULTS_DIR', '/content/drive/MyDrive/SIH/privEmbeddings/2026-11-16/results_attack')
ANALYSIS_DIR = os.path.join(os.path.dirname(RESULTS_DIR.rstrip('/\\')), 'analysis_attack')
SAVE_OUTPUTS = True   # salva figuras e tabelas em ANALYSIS_DIR
N_BOOT = 1000         # reamostragens bootstrap para os ICs de 95%
RNG = np.random.default_rng(0)

FIG_DIR = os.path.join(ANALYSIS_DIR, 'figs')
TAB_DIR = os.path.join(ANALYSIS_DIR, 'tables')
if SAVE_OUTPUTS:
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(TAB_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 11, 'axes.labelsize': 10})


def savefig(name):
    plt.tight_layout()
    if SAVE_OUTPUTS:
        plt.savefig(os.path.join(FIG_DIR, name + '.png'), dpi=150, bbox_inches='tight')
    plt.show()


def export(table, name):
    if SAVE_OUTPUTS:
        table.to_csv(os.path.join(TAB_DIR, name + '.csv'))
    return table

print('RESULTS_DIR:', RESULTS_DIR, '| existe:', os.path.isdir(RESULTS_DIR))

## 1. Carga e validação

In [ ]:
def load_runs(root):
    run_rows, frames = [], []
    for dirpath, dirs, files in sorted(os.walk(root)):
        csvs = [f for f in files if f.endswith('.csv')]
        jsons = [f for f in files if f.endswith('.json')]
        if not csvs or not jsons:
            continue
        rel = os.path.relpath(dirpath, root).replace('\\', '/')
        js = 'summary.json' if 'summary.json' in jsons else jsons[0]
        cs = 'sentence_results.csv' if 'sentence_results.csv' in csvs else csvs[0]
        with open(os.path.join(dirpath, js), encoding='utf-8') as f:
            s = json.load(f)
        row = {k: v for k, v in s.items() if not isinstance(v, dict)}
        row.update(s.get('test_results') or {})
        row['cosine'] = (s.get('align_metrics') or {}).get('X_Y_test_COS')
        row['rel_path'] = rel
        run_rows.append(row)
        d = pd.read_csv(os.path.join(dirpath, cs))
        d['rel_path'] = rel
        frames.append(d)
    if not run_rows:
        raise FileNotFoundError(f'Nenhum par summary.json + csv encontrado em {root}')
    return pd.DataFrame(run_rows), pd.concat(frames, ignore_index=True)


def eps_from_path(p):
    m = re.search(r'eps(\d+)p(\d+)', p)
    return float(f'{m[1]}.{m[2]}') if m else np.nan


def add_keys(d):
    d['eps'] = pd.to_numeric(d['epsilon'], errors='coerce')
    d['eps'] = d['eps'].fillna(d['rel_path'].map(eps_from_path))
    d['n'] = d['train_samples'].astype(int)
    d['cond'] = np.where(d['source'] == 'bert_pretrained', 'BERT', 'ε=' + d['eps'].map('{:g}'.format))
    d.loc[d['source'] == 'bert_pretrained', 'eps'] = np.nan
    return d


runs, df = load_runs(RESULTS_DIR)
runs, df = add_keys(runs), add_keys(df)

dped_eps = sorted(runs.loc[runs.source == 'dped', 'eps'].dropna().unique(), reverse=True)
COND_ORDER = [f'ε={e:g}' for e in dped_eps] + (['BERT'] if (runs.source == 'bert_pretrained').any() else [])
DPED_CONDS = [c for c in COND_ORDER if c != 'BERT']
N_LIST = [int(n) for n in sorted(runs.n.unique())]
cm = plt.cm.viridis(np.linspace(0.05, 0.9, max(len(DPED_CONDS), 1)))
PAL = {c: cm[i] for i, c in enumerate(DPED_CONDS)}
PAL['BERT'] = '#555555'
NPAL = dict(zip(N_LIST, plt.cm.plasma(np.linspace(0.05, 0.85, len(N_LIST)))))

for d in (runs, df):
    d['cond'] = pd.Categorical(d['cond'], categories=COND_ORDER, ordered=True)

df = df.sort_values(['cond', 'n', 'sentence_id']).reset_index(drop=True)
runs = runs.sort_values(['cond', 'n']).reset_index(drop=True)

print(f'{len(runs)} execuções | {len(df)} linhas (frase × execução)')
print('Condições:', COND_ORDER)
print('n:', N_LIST)
print('ridge:', [float(r) for r in sorted(df.ridge.unique())], '| seed:', [int(s) for s in sorted(df.seed.unique())])
if df.groupby(['cond', 'n'], observed=True).ngroups != len(runs):
    print('ATENÇÃO: há mais de uma execução por (condição, n) (ridge/seed distintos). As agregações misturam essas execuções.')
display(runs.head(3))

In [ ]:
cover = df.groupby(['cond', 'n'], observed=True).sentence_id.nunique().unstack('n').reindex(COND_ORDER)
fig, ax = plt.subplots(figsize=(1.0 * len(N_LIST) + 2, 0.6 * len(COND_ORDER) + 1.5))
sns.heatmap(cover, annot=True, fmt='.0f', cmap='Blues', cbar=False, ax=ax)
ax.set_title('Cobertura: frases de teste distintas por (condição, n)')
ax.set_xlabel('n (pares vazados)'); ax.set_ylabel('')
savefig('00_cobertura')

missing = [(c, n) for c in COND_ORDER for n in N_LIST if pd.isna(cover.loc[c].get(n))]
print('Combinações ausentes:', missing or 'nenhuma')
print('Frases de teste em comum entre todas as execuções:', len(set.intersection(*[set(g.sentence_id) for _, g in df.groupby('rel_path')])))
print('Elegíveis (train/test):', runs.eligible_train.unique(), runs.eligible_test.unique(), '| dims X→Y:', runs.source_dim.unique(), runs.target_dim.unique())

## 2. Engenharia de atributos

Por frase: comprimento (palavras) da referência e da predição, precisão/recall/F1 de unigramas, recall de palavras de **conteúdo** vs *stopwords*, repetição na predição e uma **linha de base de acaso**: o F1 de unigramas da predição de *outra* frase da mesma execução (deslocamentos cíclicos). `f1 - f1_chance` mede quanto da sobreposição lexical é informação vazada, e não vocabulário genérico (artigos, preposições).

In [ ]:
STOP = set("""a an the and or but if of to in on at by for with from as is are was were be been being it its this that these
those he she they we you i his her their our your not no nor so than too very can will just do does did has have had
s t n re ve ll d m there here what which who whom when where how all any both each more most other some such only own
same into over about out up off again further then once""".split())


def toks(s):
    return re.findall(r"[a-z0-9]+", str(s).lower())


def f1_pr(p, r):
    ov = sum((Counter(p) & Counter(r)).values())
    prec = ov / len(p) if p else 0.0
    rec = ov / len(r) if r else 0.0
    return prec, rec, (2 * prec * rec / (prec + rec) if prec + rec else 0.0)


df['p_tok'] = df.prediction.fillna('').map(toks)
df['r_tok'] = df.reference.map(toks)
df[['prec', 'rec', 'f1']] = [f1_pr(p, r) for p, r in zip(df.p_tok, df.r_tok)]
df['ref_len'] = df.r_tok.map(len)
df['pred_len'] = df.p_tok.map(len)
df['len_diff'] = df.pred_len - df.ref_len
df['len_ratio'] = df.pred_len / df.ref_len.replace(0, np.nan)


def content_rec(p, r, stop):
    ref = {w for w in r if (w in STOP) == stop}
    return len(ref & set(p)) / len(ref) if ref else np.nan


df['content_rec'] = [content_rec(p, r, False) for p, r in zip(df.p_tok, df.r_tok)]
df['stop_rec'] = [content_rec(p, r, True) for p, r in zip(df.p_tok, df.r_tok)]
df['distinct1'] = [len(set(p)) / len(p) if p else np.nan for p in df.p_tok]
df['rep_bigram'] = [1 - len(set(zip(p, p[1:]))) / (len(p) - 1) if len(p) > 1 else np.nan for p in df.p_tok]


def chance_f1(g, shifts=5):
    preds, refs, m = g.p_tok.tolist(), g.r_tok.tolist(), len(g)
    if m < 2:
        return pd.Series(np.nan, index=g.index)
    sh = RNG.choice(np.arange(1, m), size=min(shifts, m - 1), replace=False)
    out = np.mean([[f1_pr(preds[(i + s) % m], refs[i])[2] for i in range(m)] for s in sh], axis=0)
    return pd.Series(out, index=g.index)


df['f1_chance'] = df.groupby('rel_path', group_keys=False).apply(chance_f1)
df['excess_f1'] = df.f1 - df.f1_chance

ref_u = df.drop_duplicates('sentence_id')
edges = sorted({int(x) for x in np.quantile(ref_u.ref_len, [.25, .5, .75])})
LEN_LABELS = [f'≤{edges[0]}'] + [f'{a + 1}–{b}' for a, b in zip(edges[:-1], edges[1:])] + [f'>{edges[-1]}']
df['len_bin'] = pd.cut(df.ref_len, [-1] + edges + [10 ** 6], labels=LEN_LABELS)

print('Concordância ROUGE-1 (CSV) vs F1 de unigramas próprio: Spearman =', round(stats.spearmanr(df.rouge1, df.f1)[0], 3))
print('Comprimento da referência (palavras): média %.1f | mediana %.0f | min %d | max %d' % (ref_u.ref_len.mean(), ref_u.ref_len.median(), ref_u.ref_len.min(), ref_u.ref_len.max()))
print('Faixas de tamanho:', LEN_LABELS)

In [ ]:
def boot_ci(x, alpha=.05):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if len(x) < 2:
        return np.nan, np.nan
    means = x[RNG.integers(0, len(x), (N_BOOT, len(x)))].mean(1)
    return tuple(np.quantile(means, [alpha / 2, 1 - alpha / 2]))


def agg_ci(d, cols, by=('cond', 'n')):
    by = list(by)
    rows = []
    for key, g in d.groupby(by, observed=True):
        r = dict(zip(by, key if isinstance(key, tuple) else (key,)))
        for c in cols:
            lo, hi = boot_ci(g[c])
            r[c], r[c + '_lo'], r[c + '_hi'] = g[c].mean(), lo, hi
        rows.append(r)
    return pd.DataFrame(rows)


SENT_METRICS = {
    'rouge1': 'ROUGE-1 F1', 'rougeL': 'ROUGE-L F1', 'cosine': 'Cosseno (X·T vs Y)',
    'f1': 'F1 de unigramas', 'content_rec': 'Recall de palavras de conteúdo', 'excess_f1': 'F1 acima do acaso',
}
A = agg_ci(df, list(SENT_METRICS))
export(A.set_index(['cond', 'n']), 'agregado_sentenca_cond_n')


def set_log_n(ax):
    ax.set_xscale('log')
    ax.set_xticks(N_LIST)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.set_minor_locator(NullLocator())
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    ax.set_xlabel('n (pares vazados)')

## 3. Panorama: todas as métricas do sumário em (condição × n)

Cada painel é uma métrica do `summary.json`. Leia as colunas para o efeito de **n** e as linhas para o efeito de **ε** (da menor para a maior privacidade, de cima para baixo; o baseline BERT, sem privacidade, por último).

In [ ]:
SUM_METRICS = [m for m in ['bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4', 'rouge1', 'rouge2', 'rougeL', 'exact_match', 'cosine'] if m in runs]
rows_ = (len(SUM_METRICS) + 1) // 2
fig, axes = plt.subplots(rows_, 2, figsize=(13, 3.0 * rows_))
for ax, m in zip(axes.ravel(), SUM_METRICS):
    pv = runs.pivot_table(index='cond', columns='n', values=m, observed=True).reindex(COND_ORDER)
    sns.heatmap(pv, annot=True, fmt='.3g', cmap='viridis', cbar=False, ax=ax)
    ax.set_title(m); ax.set_xlabel('n'); ax.set_ylabel('')
for ax in axes.ravel()[len(SUM_METRICS):]:
    ax.axis('off')
fig.suptitle('Métricas do sumário por (condição, n)', y=1.0, fontsize=13)
savefig('01_panorama_heatmaps')
export(runs.set_index(['cond', 'n'])[SUM_METRICS], 'sumario_metricas')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, m in zip(axes.ravel(), SUM_METRICS):
    for c in COND_ORDER:
        s = runs[runs.cond == c].sort_values('n')
        ax.plot(s.n, s[m], marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
    set_log_n(ax); ax.set_title(m)
for ax in axes.ravel()[len(SUM_METRICS):]:
    ax.axis('off')
axes.ravel()[0].legend(title='condição')
fig.suptitle('Métricas do sumário vs n (escala log)', fontsize=13)
savefig('02_metricas_vs_n')

## 4. Tendências com intervalo de confiança (nível de frase)

Usa as 200 frases de teste de cada execução e **IC bootstrap de 95%** da média. As bandas largas para n pequeno indicam que a diferença entre configurações pode ser ruído; confirme nos testes pareados da seção 6.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
for ax, (m, label) in zip(axes.ravel(), SENT_METRICS.items()):
    for c in COND_ORDER:
        s = A[A.cond == c].sort_values('n')
        ax.plot(s.n, s[m], marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
        ax.fill_between(s.n, s[m + '_lo'], s[m + '_hi'], color=PAL[c], alpha=.15)
    set_log_n(ax); ax.set_title(label); ax.set_ylabel('média ± IC95%')
axes[0, 0].legend(title='condição')
savefig('03_tendencia_vs_n_ci')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
xs = np.arange(len(COND_ORDER))
for ax, (m, label) in zip(axes.ravel(), SENT_METRICS.items()):
    for n in N_LIST:
        s = A[A.n == n].set_index('cond').reindex(COND_ORDER)
        ax.errorbar(xs + (N_LIST.index(n) - len(N_LIST) / 2) * 0.03, s[m], yerr=[s[m] - s[m + '_lo'], s[m + '_hi'] - s[m]],
                    marker='o', ms=4, color=NPAL[n], label=f'n={n}', capsize=2, lw=1.2)
    ax.set_xticks(xs); ax.set_xticklabels(COND_ORDER); ax.set_title(label)
    ax.set_xlabel('mais ε (menos privacidade)  →  mais privacidade  →  baseline sem privacidade')
axes[0, 0].legend(ncol=2, fontsize=8)
savefig('04_tendencia_vs_epsilon_ci')

In [ ]:
d_pv = lambda m: A.pivot(index='cond', columns='n', values=m).reindex(COND_ORDER)
fig, axes = plt.subplots(2, 3, figsize=(17, 7))
for ax, (m, label) in zip(axes.ravel(), SENT_METRICS.items()):
    sns.heatmap(d_pv(m), annot=True, fmt='.3f', cmap='magma', ax=ax, cbar=False)
    ax.set_title(label); ax.set_xlabel('n'); ax.set_ylabel('')
savefig('05_heatmaps_sentenca')

In [ ]:
# Sensibilidade: o que muda ao variar ε (mantendo n) e ao variar n (mantendo ε)
KEY = [m for m in ['rouge1', 'rougeL', 'bleu1', 'bleu', 'cosine'] if m in runs]
pm = runs.pivot_table(index='cond', columns='n', values=KEY, observed=True)
if len(DPED_CONDS) >= 2:
    hi_c, lo_c = DPED_CONDS[0], DPED_CONDS[-1]
    eps_eff = pd.DataFrame({m: pm[m].loc[lo_c] / pm[m].loc[hi_c] for m in KEY})
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.heatmap(eps_eff.T, annot=True, fmt='.2f', cmap='RdYlGn', center=1, ax=axes[0])
    axes[0].set_title(f'Razão métrica({lo_c}) / métrica({hi_c}) por n\n(<1: mais privacidade reduz o ataque)')
    axes[0].set_xlabel('n')
n_eff = pd.DataFrame({m: pm[m][N_LIST[-1]] / pm[m][N_LIST[0]] for m in KEY}).reindex(DPED_CONDS)
sns.heatmap(n_eff.T, annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1])
axes[1].set_title(f'Razão métrica(n={N_LIST[-1]}) / métrica(n={N_LIST[0]}) por ε')
savefig('06_razoes_sensibilidade')

## 5. Alinhamento (cosseno) vs qualidade textual

O cosseno mede o quão perto o vetor alinhado `X·T` está do `Y` verdadeiro **antes** da decodificação. O ALGEN decodifica a partir de `X·T`, então um cosseno alto não garante texto fiel. Aqui vemos o quanto o cosseno prevê ROUGE.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for c in COND_ORDER:
    s = runs[runs.cond == c]
    axes[0].scatter(s.cosine, s.rouge1, s=25 + 12 * np.log10(s.n + 1) ** 2, color=PAL[c], label=c, alpha=.8, edgecolor='k')
axes[0].set_xlabel('cosseno médio (execução)'); axes[0].set_ylabel('ROUGE-1 (sumário)')
axes[0].set_title('Nível de execução (tamanho ∝ log n)'); axes[0].legend()

sc = axes[1].scatter(df.cosine, df.rouge1, c=np.log10(df.n), cmap='plasma', s=6, alpha=.25)
plt.colorbar(sc, ax=axes[1], label='log10 n')
axes[1].set_xlabel('cosseno (frase)'); axes[1].set_ylabel('ROUGE-1 (frase)'); axes[1].set_title('Nível de frase')

sp = df.groupby(['cond', 'n'], observed=True).apply(lambda g: stats.spearmanr(g.cosine, g.rouge1)[0]).unstack('n').reindex(COND_ORDER)
sns.heatmap(sp, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[2])
axes[2].set_title('Spearman(cosseno, ROUGE-1) entre frases'); axes[2].set_ylabel('')
savefig('07_cosseno_vs_rouge')
print('Spearman global cosseno × rouge1:', round(stats.spearmanr(df.cosine, df.rouge1)[0], 3))

In [ ]:
# O sumário reproduz a média por frase do CSV?
chk = runs.merge(df.groupby('rel_path')[['rouge1', 'rougeL', 'cosine']].mean().add_suffix('_csv').reset_index(), on='rel_path')
for m in ['rouge1', 'rougeL', 'cosine']:
    print(f'{m}: |sumário - média CSV| médio = {(chk[m] - chk[m + "_csv"]).abs().mean():.4f}')

## 6. Distribuições e testes pareados

As mesmas frases de teste são usadas em todas as execuções, então comparações são **pareadas por `sentence_id`** (Wilcoxon). Valores-p corrigidos por Bonferroni dentro de cada família de testes. Há uma única semente (0): as diferenças refletem a variação das frases de teste, não a variação de amostragem dos pares de treino.

In [ ]:
fig, axes = plt.subplots(len(COND_ORDER), 3, figsize=(17, 2.9 * len(COND_ORDER)), squeeze=False, sharex='col')
for i, c in enumerate(COND_ORDER):
    g = df[df.cond == c]
    for j, m in enumerate(['rouge1', 'rougeL', 'cosine']):
        sns.boxplot(data=g, x='n', y=m, ax=axes[i, j], color=PAL[c], fliersize=1.5, linewidth=.8)
        axes[i, j].set_ylabel(c if j == 0 else ''); axes[i, j].set_title(m if i == 0 else '')
savefig('08_boxplots')

In [ ]:
focus_ns = [N_LIST[0], N_LIST[len(N_LIST) // 2], N_LIST[-1]]
fig, axes = plt.subplots(1, len(focus_ns), figsize=(5.5 * len(focus_ns), 4.5), sharey=True)
for ax, n in zip(axes, focus_ns):
    for c in COND_ORDER:
        sns.ecdfplot(df[(df.cond == c) & (df.n == n)].rouge1, ax=ax, color=PAL[c], label=c)
    ax.set_title(f'ECDF do ROUGE-1 (n={n})'); ax.set_xlabel('ROUGE-1 por frase')
axes[0].legend()
savefig('09_ecdf_rouge1')

fig, ax = plt.subplots(figsize=(8, 4))
for c in COND_ORDER:
    sns.kdeplot(df[(df.cond == c) & (df.n == N_LIST[-1])].cosine, ax=ax, color=PAL[c], label=c, fill=True, alpha=.12)
ax.set_title(f'Distribuição do cosseno por frase (n={N_LIST[-1]})'); ax.legend()
savefig('10_kde_cosseno')

In [ ]:
def get_piv(metric):
    return df.pivot_table(index='sentence_id', columns=['cond', 'n'], values=metric, observed=True)


def stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''


def paired(x, y):
    d = (x - y).dropna()
    if len(d) < 2 or (d == 0).all():
        return np.nan, 1.0
    return d.mean(), stats.wilcoxon(d).pvalue


def paired_table(pairs, piv, index_name):
    rows = []
    for idx, col, a, b in pairs:
        diff, p = paired(piv[a], piv[b])
        rows.append((idx, col, diff, p))
    t = pd.DataFrame(rows, columns=[index_name, 'comparação', 'diff', 'p'])
    t['p_adj'] = np.minimum(1, t.p * len(t))
    t['txt'] = t.apply(lambda r: f'{r["diff"]:+.3f}{stars(r["p_adj"])}' if pd.notna(r['diff']) else '', axis=1)
    return t


def show_paired(t, index_name, order, title, fname, cols):
    v = t.pivot(index=index_name, columns='comparação', values='diff').reindex(order)[cols]
    a = t.pivot(index=index_name, columns='comparação', values='txt').reindex(order)[cols]
    fig, ax = plt.subplots(figsize=(1.6 * len(cols) + 2, 0.55 * len(order) + 1.8))
    sns.heatmap(v, annot=a, fmt='', cmap='RdBu', center=0, ax=ax, cbar_kws={'label': 'Δ média'})
    ax.set_title(title); ax.set_ylabel(''); ax.set_xlabel('')
    savefig(fname)


piv_r1 = get_piv('rouge1')
# (a) efeito de aumentar n, dentro de cada condição
steps = list(zip(N_LIST[:-1], N_LIST[1:]))
step_lbl = {s: f'{s[0]}→{s[1]}' for s in steps}
t_n = paired_table([(c, step_lbl[s], (c, s[1]), (c, s[0])) for c in COND_ORDER for s in steps], piv_r1, 'cond')
show_paired(t_n, 'cond', COND_ORDER, 'Δ ROUGE-1 ao aumentar n (pareado; *p<.05, **p<.01, ***p<.001, Bonferroni)', '11_teste_n', [step_lbl[s] for s in steps])
export(t_n.set_index(['cond', 'comparação']), 'teste_pareado_n_rouge1')

In [ ]:
# (b) efeito de reduzir ε (próximo ε mais privado vs anterior) em cada n
if len(DPED_CONDS) >= 2:
    trans = list(zip(DPED_CONDS[:-1], DPED_CONDS[1:]))
    tr_lbl = {t: f'{t[0]}→{t[1]}' for t in trans}
    t_e = paired_table([(n, tr_lbl[t], (t[1], n), (t[0], n)) for n in N_LIST for t in trans], piv_r1, 'n')
    show_paired(t_e, 'n', N_LIST, 'Δ ROUGE-1 ao reduzir ε (pareado; Bonferroni)', '12_teste_epsilon', [tr_lbl[t] for t in trans])
    export(t_e.set_index(['n', 'comparação']), 'teste_pareado_eps_rouge1')

## 7. Tamanho da frase

Como o ALGEN gera texto a partir de **um único vetor médio**, esperamos que a predição tenha comprimento quase fixo, independente da referência. Isso cria um viés mecânico: o *recall* cai e a *precisão* sobe em frases longas, e vice-versa nas curtas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.histplot(ref_u.ref_len, bins=30, ax=axes[0], color='steelblue')
for e in edges:
    axes[0].axvline(e, color='k', ls=':')
axes[0].set_title('Distribuição do comprimento das referências (palavras)')
for c in COND_ORDER:
    s = df[df.cond == c].groupby('n').pred_len.mean()
    axes[1].plot(s.index, s.values, marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
axes[1].axhline(ref_u.ref_len.mean(), color='k', ls=':', label='referência (média)')
set_log_n(axes[1]); axes[1].set_title('Comprimento médio da predição'); axes[1].legend(fontsize=8)
for c in COND_ORDER:
    s = df[df.cond == c].groupby('n').len_ratio.median()
    axes[2].plot(s.index, s.values, marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
axes[2].axhline(1, color='k', ls=':')
set_log_n(axes[2]); axes[2].set_title('Mediana de |pred| / |ref|')
savefig('13_comprimento_pred')

In [ ]:
FOCUS_COND = DPED_CONDS[0]   # altere para analisar outra condição
sel = df[df.cond == FOCUS_COND]
g = sns.FacetGrid(sel, col='n', col_wrap=4, height=3, sharex=True, sharey=True)
g.map_dataframe(lambda data, **kw: (plt.scatter(data.ref_len + RNG.normal(0, .15, len(data)), data.pred_len, s=10, alpha=.4, color=PAL[FOCUS_COND]),
                                    plt.plot([0, ref_u.ref_len.max()], [0, ref_u.ref_len.max()], 'k:')))
g.set_axis_labels('palavras na referência', 'palavras na predição'); g.set_titles('n={col_name}')
g.fig.suptitle(f'Comprimento da predição vs referência ({FOCUS_COND})', y=1.02)
savefig('14_scatter_comprimento')

slope = df.groupby(['cond', 'n'], observed=True).apply(lambda g: stats.linregress(g.ref_len, g.pred_len).slope).unstack('n').reindex(COND_ORDER)
corr_len = df.groupby(['cond', 'n'], observed=True).apply(lambda g: stats.spearmanr(g.ref_len, g.pred_len)[0]).unstack('n').reindex(COND_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
sns.heatmap(slope, annot=True, fmt='.2f', cmap='viridis', ax=axes[0]); axes[0].set_title('Inclinação de |pred| ~ |ref| (1 = acompanha o tamanho)')
sns.heatmap(corr_len, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1]); axes[1].set_title('Spearman(|ref|, |pred|)')
savefig('15_adaptacao_comprimento')

In [ ]:
Alen = agg_ci(df, ['rouge1', 'prec', 'rec', 'cosine', 'excess_f1'], by=('cond', 'n', 'len_bin'))
export(Alen.set_index(['cond', 'n', 'len_bin']), 'agregado_por_tamanho')
fig, axes = plt.subplots(1, len(COND_ORDER), figsize=(4.2 * len(COND_ORDER), 4), sharey=True, squeeze=False)
for ax, c in zip(axes[0], COND_ORDER):
    for b, col in zip(LEN_LABELS, sns.color_palette('flare', len(LEN_LABELS))):
        s = Alen[(Alen.cond == c) & (Alen.len_bin == b)].sort_values('n')
        ax.plot(s.n, s.rouge1, marker='o', color=col, label=b)
    set_log_n(ax); ax.set_title(c)
axes[0, 0].set_ylabel('ROUGE-1'); axes[0, 0].legend(title='|ref| (palavras)', fontsize=8)
fig.suptitle('ROUGE-1 vs n por faixa de comprimento da referência', y=1.03)
savefig('16_rouge_por_tamanho_n')

fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, (m, lab) in zip(axes.ravel(), [('rouge1', 'ROUGE-1'), ('prec', 'Precisão de unigramas'), ('rec', 'Recall de unigramas'),
                                        ('cosine', 'Cosseno'), ('excess_f1', 'F1 acima do acaso')] ):
    pv = df[df.n == N_LIST[-1]].groupby(['cond', 'len_bin'], observed=True)[m].mean().unstack('len_bin').reindex(COND_ORDER)
    sns.heatmap(pv, annot=True, fmt='.3f', cmap='mako_r', ax=ax, cbar=False); ax.set_title(f'{lab} (n={N_LIST[-1]})'); ax.set_ylabel(''); ax.set_xlabel('|ref|')
axes.ravel()[-1].axis('off')
savefig('17_metricas_por_faixa')

cl = {m: df.groupby(['cond', 'n'], observed=True).apply(lambda g: stats.spearmanr(g.ref_len, g[m])[0]).unstack('n').reindex(COND_ORDER)
      for m in ['rouge1', 'cosine', 'rec', 'prec']}
fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
for ax, (m, t) in zip(axes, cl.items()):
    sns.heatmap(t, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, cbar=False); ax.set_title(f'Spearman(|ref|, {m})')
savefig('18_correlacao_comprimento_metricas')

## 8. Nível de palavra: o que é recuperado

Para cada palavra única da referência, verifica se aparece na predição. Separa *stopwords* de palavras de conteúdo, raridade (frequência nas referências de teste) e posição na frase (o vetor é uma média de tokens e, portanto, **não carrega ordem**).

In [ ]:
docfreq = Counter(w for t in ref_u.r_tok for w in set(t))
wr_rows = []
for ix, (rt, pt) in enumerate(zip(df.r_tok, df.p_tok)):
    ps, seen, L = set(pt), set(), len(rt)
    for i, w in enumerate(rt):
        if w not in seen:
            seen.add(w)
            wr_rows.append((ix, w, w in STOP, w in ps, i / max(L - 1, 1)))
wr = pd.DataFrame(wr_rows, columns=['ix', 'word', 'is_stop', 'hit', 'pos'])
wr = wr.join(df[['cond', 'n', 'len_bin']], on='ix')
wr['kind'] = np.where(wr.is_stop, 'stopword', 'conteúdo')
wr['df_word'] = wr.word.map(docfreq)
wr['tier'] = pd.cut(wr.df_word, [0, 1, 3, 10, 10 ** 6], labels=['rara (1 frase)', '2–3 frases', '4–10 frases', 'comum (>10)'])
wr['pos_bin'] = np.minimum((wr.pos * 5).astype(int), 4)
print(len(wr), 'pares (palavra única × frase × execução)')

In [ ]:
Wk = wr.groupby(['cond', 'n', 'kind'], observed=True).hit.mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, k in zip(axes, ['conteúdo', 'stopword']):
    for c in COND_ORDER:
        s = Wk[(Wk.cond == c) & (Wk.kind == k)]
        ax.plot(s.n, s.hit, marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
    set_log_n(ax); ax.set_title(f'Taxa de recuperação: palavras de {k}')
axes[0].set_ylabel('fração de palavras da referência presentes na predição'); axes[0].legend()
savefig('19_recuperacao_conteudo_vs_stop')

Wt = wr[~wr.is_stop].groupby(['cond', 'n', 'tier'], observed=True).hit.mean().reset_index()
g = sns.relplot(data=Wt, x='n', y='hit', hue='tier', col='cond', col_order=COND_ORDER, kind='line', marker='o', height=3.4, aspect=1.0,
                palette='rocket_r', facet_kws={'sharey': True})
for ax in g.axes.ravel():
    set_log_n(ax)
g.set_axis_labels('n (pares vazados)', 'taxa de recuperação')
sns.move_legend(g, 'upper left', bbox_to_anchor=(1.0, 0.9))
g.fig.suptitle('Recuperação de palavras de conteúdo por raridade', y=1.04)
savefig('20_recuperacao_por_raridade')

In [ ]:
Wp = wr[~wr.is_stop].groupby(['cond', 'n', 'pos_bin'], observed=True).hit.mean().reset_index()
g = sns.relplot(data=Wp, x='pos_bin', y='hit', hue='n', col='cond', col_order=COND_ORDER, kind='line', marker='o', height=3.4,
                palette=NPAL, facet_kws={'sharey': True})
for ax in g.axes.ravel():
    ax.set_xticks(range(5)); ax.set_xticklabels(['início', '', 'meio', '', 'fim']); ax.set_xlabel('posição na frase')
g.set_ylabels('taxa de recuperação')
sns.move_legend(g, 'upper left', bbox_to_anchor=(1.0, 0.9))
g.fig.suptitle('Recuperação de palavras de conteúdo por posição (n maior = mais escuro)', y=1.04)
savefig('21_recuperacao_por_posicao')

Wl = wr[~wr.is_stop].groupby(['cond', 'n', 'len_bin'], observed=True).hit.mean().reset_index()
pv = Wl[Wl.n == N_LIST[-1]].pivot(index='cond', columns='len_bin', values='hit').reindex(COND_ORDER)
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.heatmap(pv, annot=True, fmt='.2f', cmap='mako_r', ax=ax)
ax.set_title(f'Recuperação de conteúdo por tamanho da frase (n={N_LIST[-1]})'); ax.set_ylabel('')
savefig('22_recuperacao_por_tamanho')

In [ ]:
FOCUS_N = N_LIST[-1]   # altere para outro n
fc = df[(df.cond == FOCUS_COND) & (df.n == FOCUS_N)]
w_f = wr[(wr.cond == FOCUS_COND) & (wr.n == FOCUS_N) & ~wr.is_stop]
rate = w_f.groupby('word').hit.agg(['mean', 'count']).query('count >= 3').sort_values(['mean', 'count'], ascending=False)
halluc = Counter(w for p, r in zip(fc.p_tok, fc.r_tok) for w in p if w not in set(r))
predc = Counter(w for p in fc.p_tok for w in p)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
top = rate.head(15)[::-1]
axes[0].barh(top.index, top['mean'], color='seagreen'); axes[0].set_title(f'Palavras de conteúdo mais recuperadas (≥3 ocorrências)\n{FOCUS_COND}, n={FOCUS_N}')
bot = rate.tail(15)
axes[1].barh(bot.index[::-1], bot['mean'][::-1], color='firebrick'); axes[1].set_title('Menos recuperadas (≥3 ocorrências)')
hh = halluc.most_common(15)[::-1]
axes[2].barh([w for w, _ in hh], [c for _, c in hh], color='darkorange'); axes[2].set_title('Palavras mais "alucinadas" (na predição, ausentes da referência)')
savefig('23_palavras_recuperadas_e_alucinadas')

In [ ]:
# O vocabulário previsto se concentra em poucas palavras quando n é pequeno?
TOPW = [w for w, _ in Counter(w for p in df[(df.cond == FOCUS_COND)].p_tok for w in p).most_common(15)]
rel = pd.DataFrame({n: pd.Series(Counter(w for p in df[(df.cond == FOCUS_COND) & (df.n == n)].p_tok for w in p)).reindex(TOPW).fillna(0)
                    / max(1, sum(len(p) for p in df[(df.cond == FOCUS_COND) & (df.n == n)].p_tok)) * 100 for n in N_LIST})
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(rel, annot=True, fmt='.1f', cmap='rocket_r', ax=ax, cbar_kws={'label': '% dos tokens previstos'})
ax.set_title(f'Palavras mais previstas ({FOCUS_COND}) por n'); ax.set_xlabel('n')
savefig('24_palavras_previstas_por_n')

## 9. Diversidade e colapso de modo

Com poucos pares, o mapeamento X→Y é mal determinado e o decodificador tende a gerar frases genéricas e parecidas entre si, ignorando a entrada.

In [ ]:
def div_stats(g):
    allt = [w for p in g.p_tok for w in p]
    c = Counter(allt)
    ref_vocab = {w for r in g.r_tok for w in r}
    return pd.Series({
        'vocab_pred': len(c),
        'top20_share': sum(v for _, v in c.most_common(20)) / max(1, len(allt)),
        'unique_pred_frac': g.prediction.nunique() / len(g),
        'ref_vocab_coverage': len(ref_vocab & set(c)) / len(ref_vocab),
        'distinct1': g.distinct1.mean(),
        'rep_bigram': g.rep_bigram.mean(),
    })


div = df.groupby(['cond', 'n'], observed=True).apply(div_stats).reset_index()
export(div.set_index(['cond', 'n']), 'diversidade')
labels = {'vocab_pred': 'Tamanho do vocabulário previsto', 'top20_share': 'Fração dos tokens nas 20 palavras mais usadas',
          'unique_pred_frac': 'Fração de predições distintas', 'ref_vocab_coverage': 'Cobertura do vocabulário das referências',
          'distinct1': 'Distinct-1 médio (por predição)', 'rep_bigram': 'Bigramas repetidos (por predição)'}
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, (m, lab) in zip(axes.ravel(), labels.items()):
    for c in COND_ORDER:
        s = div[div.cond == c]
        ax.plot(s.n, s[m], marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
    set_log_n(ax); ax.set_title(lab)
axes[0, 0].legend()
savefig('25_diversidade')

## 10. Dificuldade por frase e exemplos

Algumas frases são intrinsecamente mais fáceis (por exemplo, palavras raras e distintivas). Aqui medimos o quanto a dificuldade é **consistente entre configurações**.

In [ ]:
piv_all = get_piv('rouge1')
piv_all = piv_all.loc[piv_all.mean(1).sort_values(ascending=False).index]
cols = [(c, n) for c in COND_ORDER for n in N_LIST if (c, n) in piv_all.columns]
fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(piv_all[cols], cmap='viridis', ax=ax, yticklabels=False, xticklabels=[f'{c} n={n}' for c, n in cols], cbar_kws={'label': 'ROUGE-1'})
ax.set_title('ROUGE-1 por frase (linhas, ordenadas da mais fácil à mais difícil) × configuração'); ax.set_ylabel('frases de teste')
savefig('26_frase_x_configuracao')

cc = piv_all[cols].corr(method='spearman')
cc.index = cc.columns = [f'{c} n={n}' for c, n in cols]
g = sns.clustermap(cc, cmap='vlag', center=0, figsize=(11, 11), xticklabels=True, yticklabels=True)
g.fig.suptitle('Correlação da dificuldade por frase entre configurações (Spearman)', y=1.01)
if SAVE_OUTPUTS:
    g.savefig(os.path.join(FIG_DIR, '27_clustermap_dificuldade.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fácil/difícil: o que as frases mais fáceis têm em comum?
sent = piv_all.mean(1).rename('rouge1_media').to_frame().join(ref_u.set_index('sentence_id')[['reference', 'ref_len']])
sent['frac_stop'] = [np.mean([w in STOP for w in toks(r)]) if toks(r) else np.nan for r in sent.reference]
sent['rare_frac'] = [np.mean([docfreq[w] == 1 for w in toks(r) if w not in STOP]) if [w for w in toks(r) if w not in STOP] else np.nan for r in sent.reference]
sent['avg_word_len'] = [np.mean([len(w) for w in toks(r)]) if toks(r) else np.nan for r in sent.reference]
feat = ['ref_len', 'frac_stop', 'rare_frac', 'avg_word_len']
cor = pd.Series({f: stats.spearmanr(sent[f], sent.rouge1_media, nan_policy='omit')[0] for f in feat})
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
cor.plot.barh(ax=axes[0], color='slateblue'); axes[0].set_title('Spearman com ROUGE-1 médio da frase'); axes[0].axvline(0, color='k')
sns.regplot(data=sent, x='frac_stop', y='rouge1_media', ax=axes[1], scatter_kws={'s': 12, 'alpha': .5})
axes[1].set_title('Fração de stopwords na referência vs ROUGE-1 médio')
sns.regplot(data=sent, x='ref_len', y='rouge1_media', ax=axes[2], scatter_kws={'s': 12, 'alpha': .5})
axes[2].set_title('Comprimento da referência vs ROUGE-1 médio')
savefig('28_fatores_dificuldade')
display(sent.head(8)[['rouge1_media', 'ref_len', 'reference']].style.set_caption('Frases mais fáceis (média sobre configurações)'))
display(sent.tail(8)[['rouge1_media', 'ref_len', 'reference']].style.set_caption('Frases mais difíceis'))

In [ ]:
pd.set_option('display.max_colwidth', 120)
ex = df[(df.cond == FOCUS_COND) & (df.n == FOCUS_N)].sort_values('rouge1', ascending=False)
display(ex.head(5)[['sentence_id', 'rouge1', 'cosine', 'reference', 'prediction']].style.set_caption(f'Melhores reconstruções: {FOCUS_COND}, n={FOCUS_N}'))
display(ex.tail(5)[['sentence_id', 'rouge1', 'cosine', 'reference', 'prediction']].style.set_caption('Piores reconstruções'))

# Evolução de uma mesma frase (comprimento mediano) ao aumentar n, e ao reduzir ε
mid = ref_u.iloc[(ref_u.ref_len - ref_u.ref_len.median()).abs().argsort()[:1]].sentence_id.iloc[0]
print('Frase', mid, '->', df[df.sentence_id == mid].reference.iloc[0])
display(df[(df.sentence_id == mid) & (df.cond == FOCUS_COND)].sort_values('n')[['n', 'rouge1', 'cosine', 'prediction']].set_index('n'))
display(df[(df.sentence_id == mid) & (df.n == FOCUS_N)].sort_values('cond')[['cond', 'rouge1', 'cosine', 'prediction']].set_index('cond'))

## 11. Trade-off privacidade × vazamento

Três ângulos: (a) inclinação por década de n em cada ε; (b) regressão `métrica ~ log10(n) + log2(ε) + interação` nas execuções DPED; (c) decomposição de variância em nível de frase (frase vs n vs ε); (d) n mínimo para atingir um limiar de ataque.

In [ ]:
rows = []
for c in COND_ORDER:
    for m in [x for x in ['rouge1', 'rougeL', 'bleu1', 'cosine'] if x in runs]:
        s = runs[runs.cond == c]
        if len(s) >= 3:
            lr = stats.linregress(np.log10(s.n), s[m])
            rows.append((c, m, lr.slope, lr.rvalue ** 2, lr.pvalue))
slopes = pd.DataFrame(rows, columns=['cond', 'metrica', 'ganho_por_decada_n', 'R2', 'p']).pivot(index='cond', columns='metrica', values=['ganho_por_decada_n', 'R2']).reindex(COND_ORDER)
display(export(slopes, 'inclinacao_log_n').round(4))

# Retornos decrescentes: ganho marginal entre n consecutivos (ROUGE-1)
gain = runs.pivot_table(index='cond', columns='n', values='rouge1', observed=True).reindex(COND_ORDER).diff(axis=1).iloc[:, 1:]
fig, ax = plt.subplots(figsize=(9, 3.6))
sns.heatmap(gain, annot=True, fmt='+.3f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Ganho de ROUGE-1 ao passar de um n para o seguinte'); ax.set_ylabel('')
savefig('29_retornos_decrescentes')

In [ ]:
dp = runs[runs.source == 'dped']
if dp.eps.nunique() >= 2 and dp.n.nunique() >= 2:
    ln, le = np.log10(dp.n.values), np.log2(dp.eps.values)
    ln, le = ln - ln.mean(), le - le.mean()
    X = np.column_stack([np.ones(len(dp)), ln, le, ln * le])
    names = ['intercepto', 'log10(n)', 'log2(ε)', 'log10(n)×log2(ε)']
    out = {}
    for m in [x for x in ['rouge1', 'rougeL', 'bleu1', 'cosine'] if x in dp]:
        y = dp[m].values.astype(float)
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        res = y - X @ beta
        dof = len(y) - X.shape[1]
        se = np.sqrt(np.diag(res @ res / dof * np.linalg.inv(X.T @ X)))
        out[m] = pd.DataFrame({'coef': beta, 'ep': se, 't': beta / se}, index=names).stack()
        out[m][('R2', '')] = 1 - res @ res / ((y - y.mean()) @ (y - y.mean()))
    display(export(pd.DataFrame(out).T, 'regressao_dped').round(4))
    print('Leitura: coef de log10(n) = ganho por década de n; coef de log2(ε) = ganho ao dobrar ε (variáveis centradas).')

In [ ]:
d = df[df.source == 'dped']
dec = pd.DataFrame()


def ss_share(d, y):
    gm = d[y].mean(); sst = ((d[y] - gm) ** 2).sum()
    def ss(keys):
        t = d.groupby(keys, observed=True)[y].agg(['mean', 'count'])
        return (t['count'] * (t['mean'] - gm) ** 2).sum()
    s_n, s_e, s_cell = ss('n'), ss('eps'), ss(['n', 'eps'])
    parts = {'frase': ss('sentence_id'), 'n (vazados)': s_n, 'ε': s_e, 'n × ε': s_cell - s_n - s_e}
    parts['resíduo (frase × config.)'] = sst - sum(parts.values())
    return pd.Series(parts) / sst

if d.eps.nunique() >= 2:
    dec = pd.DataFrame({m: ss_share(d, m) for m in ['rouge1', 'rougeL', 'cosine', 'f1', 'content_rec']}).T
    ax = (dec * 100).plot.barh(stacked=True, figsize=(11, 4), colormap='Set2')
    ax.set_xlabel('% da variância (nível de frase, apenas DPED)'); ax.set_title('Quanto explicam a frase, n e ε? (aprox., desenho balanceado)')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    savefig('30_decomposicao_variancia')
    display(export((dec * 100).round(2), 'decomposicao_variancia'))

In [ ]:
# n mínimo para atingir um limiar de ROUGE-1 (interpolação linear em log10 n)
def n_for_threshold(series, thr):
    s = series.sort_index()
    ns, v = s.index.values, s.values
    hit = np.where(v >= thr)[0]
    if len(hit) == 0:
        return np.nan
    i = hit[0]
    if i == 0:
        return float(ns[0])
    f = (thr - v[i - 1]) / (v[i] - v[i - 1]) if v[i] != v[i - 1] else 1
    return 10 ** (np.log10(ns[i - 1]) + f * (np.log10(ns[i]) - np.log10(ns[i - 1])))

pr = runs.pivot_table(index='n', columns='cond', values='rouge1', observed=True)[COND_ORDER]
thrs = np.round(np.linspace(pr.min().min() + 0.01, pr.max().max() - 0.005, 8), 3)
need = pd.DataFrame({c: [n_for_threshold(pr[c], t) for t in thrs] for c in COND_ORDER}, index=thrs)
need.index.name = 'limiar ROUGE-1'
display(export(need.round(0), 'n_minimo_por_limiar'))
fig, ax = plt.subplots(figsize=(8, 4.5))
for c in COND_ORDER:
    ax.plot(need.index, need[c], marker='o', color=PAL[c], label=c, ls='--' if c == 'BERT' else '-')
ax.set_yscale('log'); ax.set_xlabel('limiar de ROUGE-1'); ax.set_ylabel('n mínimo (interpolado)')
ax.set_title('Pares vazados necessários para atingir um nível de ataque (NaN = não atingido)'); ax.legend()
savefig('31_n_minimo')

## 12. DPED vs baseline BERT

Diferença (DPED − BERT) por frase, para o mesmo n. Diferença **negativa** significa que os embeddings DPED vazam menos que os do BERT pré-treinado, ou seja, que a privacidade ajudou.

In [ ]:
if 'BERT' in COND_ORDER and DPED_CONDS:
    rows = []
    for c in DPED_CONDS:
        for n in N_LIST:
            if (c, n) in piv_r1.columns and ('BERT', n) in piv_r1.columns:
                dd = (piv_r1[(c, n)] - piv_r1[('BERT', n)]).dropna()
                lo, hi = boot_ci(dd)
                rows.append((c, n, dd.mean(), lo, hi, (dd > 0).mean() + .5 * (dd == 0).mean(), paired(piv_r1[(c, n)], piv_r1[('BERT', n)])[1]))
    vb = pd.DataFrame(rows, columns=['cond', 'n', 'delta', 'lo', 'hi', 'win_rate', 'p'])
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.6))
    for c in DPED_CONDS:
        s = vb[vb.cond == c]
        axes[0].plot(s.n, s.delta, marker='o', color=PAL[c], label=c); axes[0].fill_between(s.n, s.lo, s.hi, color=PAL[c], alpha=.15)
    axes[0].axhline(0, color='k', ls=':'); set_log_n(axes[0]); axes[0].set_title('Δ ROUGE-1 (DPED − BERT), IC95%'); axes[0].legend()
    sns.heatmap(vb.pivot(index='cond', columns='n', values='win_rate').reindex(DPED_CONDS), annot=True, fmt='.2f', cmap='RdBu', center=.5, ax=axes[1])
    axes[1].set_title('Taxa de frases em que DPED > BERT (empates = ½)'); axes[1].set_ylabel('')
    ann = vb.assign(t=lambda x: x.delta.map('{:+.3f}'.format) + (x.p * len(x)).clip(upper=1).map(stars)).pivot(index='cond', columns='n', values='t').reindex(DPED_CONDS)
    sns.heatmap(vb.pivot(index='cond', columns='n', values='delta').reindex(DPED_CONDS), annot=ann, fmt='', cmap='RdBu', center=0, ax=axes[2])
    axes[2].set_title('Δ ROUGE-1 (estrelas: Wilcoxon, Bonferroni)'); axes[2].set_ylabel('')
    savefig('32_dped_vs_bert')
    export(vb.set_index(['cond', 'n']), 'dped_vs_bert')

    # Frases e métricas em que o ataque DPED supera o BERT
    r = runs.pivot_table(index='n', columns='cond', values='rouge1', observed=True)
    display((r[DPED_CONDS].div(r['BERT'], axis=0)).round(2).style.set_caption('ROUGE-1(DPED) / ROUGE-1(BERT) por n'))
else:
    print('Baseline BERT ausente: seção ignorada.')

## 13. Achados automáticos e exportação

Resumo numérico gerado a partir dos dados (não substitui a interpretação).

In [ ]:
def trend(x, y):
    r, p = stats.spearmanr(x, y)
    return r, p

out = []
dpr = runs[runs.source == 'dped']
best = runs.loc[runs.rouge1.idxmax()]
out.append(f'- Melhor ataque (ROUGE-1 no sumário): **{best.cond}**, n={best.n}: ROUGE-1={best.rouge1:.3f}'
           + (f', BLEU-1={best.bleu1:.1f}' if 'bleu1' in runs else '') + f', cosseno={best.cosine:.3f}.')
for c in COND_ORDER:
    s = runs[runs.cond == c]
    r, p = trend(s.n, s.rouge1)
    out.append(f'- **{c}**: ROUGE-1 vai de {s.sort_values("n").rouge1.iloc[0]:.3f} (n={s.n.min()}) a {s.sort_values("n").rouge1.iloc[-1]:.3f} (n={s.n.max()}); Spearman(n, ROUGE-1)={r:+.2f}.')
if len(DPED_CONDS) >= 2:
    for n in N_LIST:
        s = dpr[dpr.n == n]
        if s.eps.nunique() >= 2:
            r, p = trend(s.eps, s.rouge1)
            out.append(f'- n={n}: Spearman(ε, ROUGE-1)={r:+.2f} (ε menor → {"menos" if r > 0 else "mais"} vazamento textual).')
if len(dec):
    out.append('- Decomposição de variância do ROUGE-1 (DPED): ' + ', '.join(f'{k} {v * 100:.1f}%' for k, v in dec.loc['rouge1'].items()) + '.')
if 'vb' in globals():
    out.append(f'- Δ ROUGE-1 (DPED − BERT) médio sobre todas as configurações: {vb.delta.mean():+.3f}; DPED supera o BERT em {100 * (vb.delta > 0).mean():.0f}% das configurações.')
display(Markdown('\n'.join(out)))

In [ ]:
export(runs.drop(columns=['rel_path']).set_index(['cond', 'n']), 'runs_consolidado')
df.drop(columns=['p_tok', 'r_tok']).to_csv(os.path.join(TAB_DIR, 'sentencas_enriquecidas.csv'), index=False) if SAVE_OUTPUTS else None
print('Saídas em:', ANALYSIS_DIR if SAVE_OUTPUTS else '(SAVE_OUTPUTS=False)')